In [1]:
import torch as t
import numpy as np

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset, random_split
from tqdm import tqdm

In [2]:
device = "cuda" if t.cuda.is_available() else "cpu"
class MyModel(t.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = t.nn.Sequential(
            t.nn.Flatten(),
            t.nn.Linear(28*28, 16),
            t.nn.ReLU(),
            t.nn.Linear(16, 10),
            t.nn.Softmax(dim=1)
        )
    def forward(self, input):
        return self.model(input)

class MySmallModel(t.nn.Module):
    def __init__(self):
        super().__init__()

        self.model = t.nn.Sequential(
            t.nn.Flatten(),
            t.nn.Linear(28*28, 10),
            t.nn.Softmax(dim=1)
        )
    def forward(self, input):
        return self.model(input)

device = "cuda" if t.cuda.is_available() else "cpu"
model = MySmallModel()

In [3]:
hyperparams = {
    "lr": 1e-5,
    "batch_size" : 128,
    "num_epochs" : 1,
    "momentum" : 0.8,
}

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_set = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = t.utils.data.DataLoader(train_set, batch_size=hyperparams["batch_size"], shuffle=True)
test_loader = t.utils.data.DataLoader(test_set, batch_size=hyperparams["batch_size"], shuffle=False)

In [4]:
def count_parameters(model):
    """
    Count the total number of parameters in a PyTorch model.

    Args:
        model (torch.nn.Module): The PyTorch model.

    Returns:
        int: Total number of parameters in the model.
    """
    return sum(p.numel() for p in model.parameters())

length = count_parameters(model)
print(length)

7850


In [13]:
def generator(collection):
    for i in collection:
        yield i

def hessian(loss, model):
    x = t.autograd.grad(loss, model.parameters(), create_graph=True)

    for param_matrix in x:
        if (len(param_matrix.size()) == 2):
            for i in range(param_matrix.size()[0]):
                for j in range(param_matrix.size()[1]):
                    all_y = t.autograd.grad(param_matrix[i][j], model.parameters(), retain_graph=True)
                    for y in all_y:
                        if (len(y.size()) == 2):
                            for yi in y:
                                for yij in yi:
                                    yield yij
                        else:
                            for yi in y:
                                yield yi

        else:
            for i in range(param_matrix.size()[0]):
                all_y = t.autograd.grad(param_matrix[i], model.parameters(), retain_graph=True)
                for y in all_y:
                    if (len(y.size()) == 2):
                        for yi in y:
                            for yij in yi:
                                yield yij
                    else:
                        for yi in y:
                            yield yi



In [14]:
LossFn = t.nn.CrossEntropyLoss()
optimizer = t.optim.SGD(model.parameters(), lr=hyperparams["lr"], momentum=hyperparams["momentum"])
most_recent_grad = []


def train_one_epoch(model, train_loader, optimizer, criterions):
    i = 0
    #model.train() # i havent included dropout so shld be unnecessary
    for image, label in tqdm(train_loader):
        optimizer.zero_grad()

        image2, label2 = image.to(device), label.to(device)
        output = model(image2)

        loss = LossFn(output, label2)
        loss.backward(retain_graph=True)
        if i == 468:
            return hessian(loss, model)

        optimizer.step()
        i += 1


hessian_generator = train_one_epoch(model, train_loader, optimizer, LossFn)



100%|█████████▉| 468/469 [00:15<00:00, 29.76it/s]


In [8]:
7850 * 7850

61622500

In [15]:
x = 0
for i in hessian_generator:
    # i is a hessian element. Use striding techniques to get the elements in the matrix
    x += 1

print(x)


61622500
